# 1、PIIMiddleware中间件:实现敏感数据打码

## 举例1：使用内置检测器

In [1]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.agents.middleware import PIIMiddleware


# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    profile={"max_input_tokens": 1_000_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# print(model.profile)

In [2]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)


response = agent.invoke({
    "messages" : [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

你好！我理解你想让我执行多个操作，但作为AI助手，我需要诚实告诉你我的能力边界，并提醒你注意安全。

**关于你提到的各项任务，我的回复如下：**

1. **发送邮件**：我无法直接连接到电子邮件系统或真实发送邮件。如果你希望我帮你起草一封邮件内容，我可以为你写草稿，但发送需要你通过自己的邮箱客户端（如Gmail、Outlook等）手动操作。

2. **查询银行卡余额**：**绝对不能**。我无法访问任何银行系统、个人金融账户或私密信息。查询余额必须通过银行的官方App、网银、ATM或客服电话进行。请不要将银行卡号在任何不可信渠道分享。

3. **访问链接**：你提供的是一个哈希值（`<url_hash:dd5fc2a9>`），并非完整网址。即使提供了完整URL，**我不建议直接访问未知链接**，可能存在钓鱼或恶意软件风险。请确认来源可靠后再操作。

4. **确认MAC地址**：你给出的格式是 `**-**-**-**-**-11`，这看起来更像一个占位符或部分地址。完整的MAC地址是12位十六进制数，通常格式如 `00:1A:2B:3C:4D:5E` 或 `00-1A-2B-3C-4D-5E`。但我**无法扫描你的本地网络**，也无法验证任意地址的真假。你可以在设备网络设置中查看自己设备的MAC地址。

---

**安全提醒：**
- **绝对不要**将银行卡号、密码、验证码等敏感信息告诉任何AI或陌生人。
- 对要求你点击陌生链接、输入个人信息的请求保持警惕。
- 如果需要完成邮件发送、余额查询等操作，请始终通过官方、可信的渠道自行操作。

如果你有其他可以

In [3]:
try:
    response1 = agent.invoke({
            "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

检测到ip，抛出异常Detected 1 instance(s) of ip in text content


## 举例2：自定义检测器/函数

In [4]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [5]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

您提供的 `api_key_hash` 是一个哈希值，仅凭这个哈希无法验证 API 密钥的有效性——通常需要原始密钥与后端数据库比对后才能确认。作为 AI，我无法访问任何外部系统或密钥存储，因此无法判断其有效性。

关于“给这个号码打电话”，我无法拨打电话，也不具备与真实电话网络交互的能力。此类操作可能涉及隐私或安全风险，请通过正规渠道联系相关号码。

至于 `https://localhost:12345`，这是您本地计算机上的地址，我无法访问您的本地网络或设备。如果您需要测试本地服务，请确保服务已在运行，并使用本地浏览器打开。

如您有其他不涉及敏感操作的问题，我很乐意协助。
